# Task 8b: GPT-only and guided rows under the 5-trial protocol (GPU)

18 extra rows x 5 trials = 90 runs, same protocol as the main table (seed fix, MolE's scaffold split, trial k = search seed and final training seed k):

- `gpt_only_<model>`: free-form GPT features alone (6 rows)
- `mole_plus_guided_<model>`: MolE + guided (schema-constrained) GPT features (6 rows)
- `guided_only_<model>`: guided GPT features alone (6 rows)

Guided inputs are `data/gpt_guided_flat/gpt_features_guided_<model>.csv`, made by `scripts/flatten_guided.py` from the six `gpt_features_4o_<model>_unified.csv` files (one schema, 46 features, the same flattening for both guided rows).

**Setup:** upload the new `gpt_mole_bundle.zip` as a new version of your dataset, attach it, Accelerator = GPU, Internet on. Run cells 1-4 to check, then **Save Version -> Save & Run All**. If the session ends early, attach this notebook's previous output and run again; finished runs are skipped.

In [ ]:
# 1. Confirm XGBoost really trains on the GPU. XGBoost silently falls back to CPU
#    when no GPU is visible, so check for that warning rather than trusting the setting.
import subprocess, sys, warnings
import numpy as np
import xgboost as xgb

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("python", sys.version.split()[0], "| xgboost", xgb.__version__)
assert int(xgb.__version__.split(".")[0]) >= 2, "device='cuda' needs xgboost >= 2"

X = np.random.rand(2000, 20)
y = (X[:, 0] > 0.5).astype(int)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    xgb.train({"tree_method": "hist", "device": "cuda", "objective": "binary:logistic"},
              xgb.DMatrix(X, label=y), 10)
fallback = [str(w.message) for w in caught if "GPU" in str(w.message)]
assert not fallback, f"XGBoost fell back to CPU - turn on a GPU accelerator.\n{fallback}"
print("GPU training OK")

In [ ]:
# 2. Dependencies. build_train.py imports rdkit (unused by these runs) and reads the
#    SF3 workbook with openpyxl; install either only if the image lacks it.
import importlib.util

for module, package in [("rdkit", "rdkit"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import pandas, sklearn
print("pandas", pandas.__version__, "| scikit-learn", sklearn.__version__)

In [ ]:
# 3. Copy the bundle to a writable folder (inputs are read-only) and work from there.
#    Also restores finished runs from a previous version's output, if one is attached.
import glob, gzip, importlib, os, shutil, sys

MARKER = "scripts/run_mole_protocol_search.py"   # present only in the Task 8 bundle
found = [os.path.dirname(os.path.dirname(h))
         for h in glob.glob("/kaggle/input/**/notebooks/build_train.py", recursive=True)]
assert found, "Bundle not found - attach the dataset made from gpt_mole_bundle.zip."
# Several dataset versions can be attached at once, so use one that has the new files
# rather than whichever the glob happens to return first.
usable = [b for b in found if os.path.exists(os.path.join(b, MARKER))]
if not usable:
    listing = "\n".join(
        f"  {b}  (run_table1.py: {os.path.getsize(os.path.join(b, 'scripts/run_table1.py')):,} bytes)"
        for b in found)
    raise AssertionError(
        "The attached dataset is an OLD version - it does not contain " + MARKER + ".\n"
        "Attached bundle(s):\n" + listing + "\n"
        "Fix: upload the new gpt_mole_bundle.zip to the dataset via New Version, then in this\n"
        "notebook's Input panel remove the dataset and add it again so the newest version is\n"
        "used (run_table1.py is then ~12,463 bytes and data/gpt_guided_flat/ exists).")
BUNDLE = usable[0]
WORK = "/kaggle/working/gpt_mole"
# dirs_exist_ok refreshes code and data in place while keeping any results/ already
# computed, so a newer dataset version takes effect without restarting the kernel.
shutil.copytree(BUNDLE, WORK, dirs_exist_ok=True)
os.chdir(WORK)
print("bundle:", BUNDLE, "-> working in", os.getcwd())
if os.path.exists("MANIFEST.txt"):
    print(open("MANIFEST.txt").read())

sys.path.insert(0, "scripts")
import run_table1
importlib.reload(run_table1)   # in case an older copy was imported earlier in this kernel

# Kaggle decompresses .gz files inside an uploaded dataset, so x.tsv.gz arrives as
# x.tsv. Re-compress them to the names the scripts expect.
needed = [run_table1.SF3, run_table1.SPLIT, run_table1.MOLE, run_table1.PRESTWICK,
          run_table1.CHEMDESC, *run_table1.FREEFORM.values(), *run_table1.GUIDED.values()]
for path in needed:
    if path.endswith(".gz") and not os.path.exists(path) and os.path.exists(path[:-3]):
        with open(path[:-3], "rb") as src, gzip.open(path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        print("re-compressed", path)
missing = [p for p in needed if not os.path.exists(p)]
assert not missing, f"Missing inputs: {missing}"
print(f"all {len(needed)} inputs present")

OUT_ROOT = "results/table1_trials"
previous = [d for d in glob.glob("/kaggle/input/**/table1_trials", recursive=True)
            if not d.startswith(BUNDLE)]
for d in previous:
    shutil.copytree(d, OUT_ROOT, dirs_exist_ok=True)
    print("restored finished runs from", d)

In [ ]:
# 4. One quick run first (trial 0, guided GPT-5.2 only) to check the pipeline.
!python -u scripts/run_table1.py --trials 0 --only guided_only_gpt-5.2 --include-guided --only-extra --out-root {OUT_ROOT} --device cuda --resume
assert _exit_code == 0, "run failed - see the output above and its run.log"

In [ ]:
# 5. Full run: 18 rows x 5 trials. Set TRIALS to a subset (e.g. "0,1,2") to split across sessions.
TRIALS = "0,1,2,3,4"
!python -u scripts/run_table1.py --trials {TRIALS} --include-gpt-only --include-guided --only-extra --out-root {OUT_ROOT} --device cuda --resume
assert _exit_code == 0, "some runs failed - see the list above and each run.log"

In [ ]:
# 6. Every run must have trained on the GPU.
logs = glob.glob(f"{OUT_ROOT}/trial_*/*/run.log")
cpu_runs = [p for p in logs if "No visible GPU" in open(p, encoding="utf-8", errors="ignore").read()]
print(f"{len(logs)} run logs checked")
assert logs, "No run logs found - the runs above did not start."
assert not cpu_runs, f"{len(cpu_runs)} runs fell back to CPU: {cpu_runs}"
print("all runs on GPU")

In [ ]:
# 7. Aggregate: mean +/- sd over trials per row -> table1.csv.
!python scripts/aggregate_table1.py --root {OUT_ROOT}
assert os.path.exists(f"{OUT_ROOT}/table1.csv"), "aggregation produced no table1.csv"

In [ ]:
# 8. Package for download (Output tab -> task8b_results.zip).
archive = shutil.make_archive("/kaggle/working/task8b_results", "zip", OUT_ROOT)
print(archive, f"{os.path.getsize(archive) / 1024 ** 2:.1f} MB")